In [ ]:

import os, re, io, sys, time, glob, shutil
import numpy as np
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


OUT_BASE = "/kaggle/working/foodevpred_v3"
IN_ROOT = "/kaggle/input"
FORCE = False                              
SEED = 42

LEN_MIN, LEN_MAX = 60, 1200


HEADROOM_MULTIPLE = 3

rng = np.random.default_rng(SEED)
FETCHED = f"{OUT_BASE}/fetched"
for d in ["fetched", "fetched/positives_raw", "final/positives",
          "final/negatives", "final/reports"]:
    os.makedirs(f"{OUT_BASE}/{d}", exist_ok=True)


def find_dir(leaf, root=IN_ROOT, required=True):
    hits = [d for d in glob.glob(f"{root}/**/{leaf}", recursive=True) if os.path.isdir(d)]
    if not hits:
        if required:
            print(f"[path] no '{leaf}/' found under {root}")
        return None
    hits.sort(key=len)
    print(f"[path] {leaf}/ -> {hits[0]}")
    return hits[0]


PROV_DIR = find_dir("provenance", required=False)          # optional

SESSION = requests.Session()
SESSION.mount("https://", HTTPAdapter(max_retries=Retry(
    total=5, backoff_factor=2, status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET", "POST"])))

API = "https://rest.uniprot.org"

FIELDS = ",".join([
    "accession", "id", "protein_name", "gene_names", "organism_name",
    "organism_id", "length", "sequence", "reviewed", "protein_existence",
    "cc_subcellular_location", "go_c", "ft_signal", "ft_transmem", "keyword",
])

ACC_RE = re.compile(r"^(?:[OPQ][0-9][A-Z0-9]{3}[0-9]"
                    r"|[A-NR-Z][0-9](?:[A-Z][A-Z0-9]{2}[0-9]){1,2})(?:-\d+)?$")

EXPECTED_SUBCLASSES = ["arabidopsis_apoplastic_EV", "brassica_apoplastic_EV",
                       "human_milk_EV", "bovine_milk_EV", "donkey_milk_EV"]


def cached(path):
    return (not FORCE) and os.path.isfile(path) and os.path.getsize(path) > 0



def stage0_load_positive_ids():
    
    EXPLICIT_PATH = "/kaggle/input/datasets/harshiikkaa/foodevpred-positives-raw"
    src_dir = EXPLICIT_PATH if os.path.isdir(EXPLICIT_PATH) else \
        find_dir("foodevpred-positives-raw", required=False)
    copied = 0
    if src_dir:
        for f in glob.glob(f"{src_dir}/*.csv"):
            dst = f"{FETCHED}/positives_raw/{os.path.basename(f)}"
            if not cached(dst):
                shutil.copy(f, dst)
                copied += 1
    print(f"[stage0] copied {copied} new CSV(s) from {src_dir or '(not found)'}")

    have = {os.path.splitext(os.path.basename(f))[0]
            for f in glob.glob(f"{FETCHED}/positives_raw/*.csv")}
    missing = [sc for sc in EXPECTED_SUBCLASSES if sc not in have]
    print(f"[stage0] {len(have)}/{len(EXPECTED_SUBCLASSES)} positive subclasses ready: "
          f"{sorted(have)}")
    if missing:
        print(f"[stage0] missing: {missing}")
        print(f"[stage0] -> upload foodevpred_v3_positives_raw.zip's CSVs as a Kaggle "
              f"dataset named 'foodevpred-positives-raw', add it to this notebook's "
              f"inputs, and re-run.")


def load_positives_raw():
    files = sorted(glob.glob(f"{FETCHED}/positives_raw/*.csv"))
    if not files:
        sys.exit("[FATAL] no files in fetched/positives_raw/ -- run stage0_load_positive_ids() first")
    return pd.concat([pd.read_csv(f, low_memory=False) for f in files], ignore_index=True)



def fetch_uniprot_pool(query, tag, page_size=500, max_pages=500):
    """Paginated UniProt fetch via /uniprotkb/search + its cursor-based Link
    header, used for every negative candidate pool. Pools can run to tens
    of thousands of rows, and a single un-paginated /uniprotkb/stream
    request for a response that large can get cut off mid-transfer, so all
    pools are fetched in small (<=500-row) pages."""
    url = f"{API}/uniprotkb/search"
    params = dict(query=query, fields=FIELDS, format="tsv", size=page_size)
    frames = []
    page = 0
    while url and page < max_pages:
        page += 1
        r = None
        for attempt in range(4):
            try:
                r = SESSION.get(url, params=params, timeout=300)
                break
            except Exception as e:
                print(f"  [{tag}] page {page} network error (attempt {attempt+1}/4): {e}")
                time.sleep(3 * (attempt + 1))
                r = None
        if r is None:
            print(f"  [{tag}] page {page} failed after 4 attempts -- stopping here "
                  f"({sum(len(f) for f in frames):,} rows collected so far)")
            break
        if r.status_code != 200:
            print(f"  [{tag}] page {page} HTTP {r.status_code}: {r.text[:200]}")
            break
        if r.text.strip():
            frames.append(pd.read_csv(io.StringIO(r.text), sep="\t", low_memory=False))
        m = re.search(r'<([^>]+)>;\s*rel="next"', r.headers.get("Link", ""))
        url = m.group(1) if m else None
        params = None  # the next URL already carries the cursor + all params
        if url is None or page % 10 == 0:
            print(f"  [{tag}] page {page}: {sum(len(f) for f in frames):,} rows so far")
        time.sleep(0.2)
    if not frames:
        return pd.DataFrame(), -1
    return pd.concat(frames, ignore_index=True), 200


def get_tsv(url, params=None, tag="", timeout=1800, retries=4):
    """Fetch a (non-paginated) UniProt TSV response, retrying on network
    errors. Used only for the idmapping results stream, bounded by the
    request chunk size (<=5000 ids)."""
    for attempt in range(retries):
        try:
            r = SESSION.get(url, params=params, timeout=timeout)
        except Exception as e:
            print(f"  [{tag}] network error (attempt {attempt+1}/{retries}): {e}")
            time.sleep(3 * (attempt + 1))
            continue
        if r.status_code != 200:
            print(f"  [{tag}] HTTP {r.status_code}: {r.text[:200]}")
            return pd.DataFrame(), r.status_code
        if not r.text.strip():
            return pd.DataFrame(), 204
        return pd.read_csv(io.StringIO(r.text), sep="\t", low_memory=False), 200
    print(f"  [{tag}] gave up after {retries} attempts")
    return pd.DataFrame(), -1


def idmap(ids, tag, chunk=5000, poll=3, max_wait=1800):
    """UniProt async ID-mapping job, submitted in chunks. A failure on one
    chunk is logged and that chunk's ids are added to `failed`; it does not
    abort the whole function.

    The POST body field is literally named "from", so it is built as a
    literal dict with a string key rather than a keyword argument.
    """
    frames, failed = [], []
    for i in range(0, len(ids), chunk):
        part = ids[i:i + chunk]
        chunk_no = i // chunk + 1
        try:
            r = SESSION.post(f"{API}/idmapping/run",
                             data={"from": "UniProtKB_AC-ID", "to": "UniProtKB",
                                   "ids": ",".join(part)}, timeout=120)
            job = r.json().get("jobId")
        except Exception as e:
            print(f"  [{tag}] chunk {chunk_no} submit failed: {e}")
            failed += list(part)
            continue
        if not job:
            print(f"  [{tag}] chunk {chunk_no} submit returned no jobId "
                  f"(HTTP {r.status_code}): {r.text[:300]}")
            failed += list(part)
            continue

        t0 = time.time()
        chunk_ok = True
        while True:
            try:
                body = SESSION.get(f"{API}/idmapping/status/{job}", timeout=120).json()
            except Exception:
                time.sleep(poll)
                continue
            state = body.get("jobStatus")
            if state in (None, "FINISHED"):
                failed += body.get("failedIds", []) or []
                break
            if state in ("ERROR", "FAILED") or time.time() - t0 > max_wait:
                print(f"  [{tag}] chunk {chunk_no} job {state or 'timeout'} -- skipping this chunk only")
                failed += list(part)
                chunk_ok = False
                break
            time.sleep(poll)
        if not chunk_ok:
            continue

        df, code = get_tsv(f"{API}/idmapping/uniprotkb/results/stream/{job}",
                           params=dict(format="tsv", fields=FIELDS), tag=tag)
        if code != 200:
            print(f"  [{tag}] chunk {chunk_no} stream failed (code {code}) -- skipping this chunk only")
            failed += list(part)
            continue
        print(f"  [{tag}] chunk {chunk_no}: {len(df):,} rows")
        frames.append(df)
        time.sleep(0.5)

    out = pd.concat(frames, ignore_index=True).drop_duplicates("Entry") \
        if frames else pd.DataFrame()
    return out, failed


ANN_OUT = f"{FETCHED}/positives_annotated.csv"
GO_EV = "(go:0070062 OR go:1903561 OR go:0031988 OR go:0097708)"

POOLS = {
    "bovine": dict(taxon=9913, matches="bovine_milk_EV",
                   query=f"(organism_id:9913) AND (reviewed:true) NOT {GO_EV}",
                   fallback=None, rationale="Swiss-Prot Bos taurus ~6k entries"),
    "human": dict(taxon=9606, matches="human_milk_EV",
                  query=f"(organism_id:9606) AND (reviewed:true) NOT {GO_EV}",
                  fallback=None, rationale="Swiss-Prot Homo sapiens ~20k entries"),
    "donkey": dict(taxon=9793, matches="donkey_milk_EV",
                   query=f"(taxonomy_id:9793) NOT {GO_EV}",
                   fallback=f"(organism_id:9793) NOT {GO_EV}",
                   rationale="Swiss-Prot E. asinus has <50 entries so TrEMBL included"),
    "arabidopsis": dict(taxon=3702, matches="arabidopsis_apoplastic_EV",
                        query=f"(taxonomy_id:3702) NOT {GO_EV}",
                        fallback=f"(organism_id:3702) NOT {GO_EV}",
                        rationale="positives mix reviewed/unreviewed; pool must include both"),
    "brassica": dict(taxon=3712, matches="brassica_apoplastic_EV",
                     query=f"(taxonomy_id:3712) NOT {GO_EV}",
                     fallback=f"(organism_id:3712) NOT {GO_EV}",
                     rationale="taxonomy_id descends to var. oleracea; organism_id does not"),
}


def stage1_uniprot():
    pos = load_positives_raw()
    print(f"[stage1] positives (from stage 0): {len(pos):,} rows")

    if cached(ANN_OUT):
        print(f"[stage1] annotation cached -> {ANN_OUT}")
        ann = pd.read_csv(ANN_OUT, low_memory=False)
    else:
        req = sorted(set(pos["uniprot_id"].dropna().astype(str)))
        print(f"[stage1] {len(req):,} unique ids to resolve")
        ann, failed = idmap(req, "positives")
        if ann.empty:
            
            print(f"[stage1] !! 0 positives resolved ({len(failed):,} failed) -- "
                  f"NOT caching this result, will retry idmap on next run")
        else:
            ann.to_csv(ANN_OUT, index=False)
            print(f"[stage1] resolved {len(ann):,} / {len(req):,} ({len(failed):,} failed)")

    ANN_KEYS = set(ann["Entry"].astype(str)) if len(ann) else set()
    if len(ann) and "Entry Name" in ann.columns:
        ANN_KEYS |= set(ann["Entry Name"].astype(str))

    EXCLUDE_ACC = set(pos["uniprot_id"].dropna().astype(str)) | ANN_KEYS
    if PROV_DIR:
        ev_path = f"{PROV_DIR}/ev_reference_accessions_exclude_from_negatives.csv"
        if os.path.isfile(ev_path):
            extra = pd.read_csv(ev_path)["uniprot_id"].dropna().astype(str).tolist()
            EXCLUDE_ACC |= set(extra)
            print(f"[stage1] +{len(extra):,} accessions from provenance exclusion list")
    print(f"[stage1] exclusion set: {len(EXCLUDE_ACC):,} accessions")

    qlog, audit = [], []
    for name, spec in POOLS.items():
        out_path = f"{FETCHED}/negative_pool_{name}.csv"
        if cached(out_path):
            print(f"[stage1] pool {name} cached")
            continue
        print(f"\n[stage1] pool {name}: {spec['query']}")
        df, code = fetch_uniprot_pool(spec["query"], tag=name)
        used, fb = spec["query"], False
        if df.empty and spec["fallback"]:
            print(f"    empty -- fallback: {spec['fallback']}")
            df, code = fetch_uniprot_pool(spec["fallback"], tag=f"{name}-fb")
            used, fb = spec["fallback"], True
        if df.empty:
            print(f"    !! {name} POOL EMPTY (code {code})")
            qlog.append(dict(pool=name, query_used=used, n_raw=0, status="FAILED"))
            continue
        n_raw = len(df)
        entry = df["Entry"].astype(str)
        ename = df["Entry Name"].astype(str) if "Entry Name" in df.columns else pd.Series("", index=df.index)
        keep = ~(entry.isin(EXCLUDE_ACC) | ename.isin(EXCLUDE_ACC))
        df = df[keep].copy()
        df["negative_pool"] = name
        df["uniprot_query"] = used
        df.to_csv(out_path, index=False)
        print(f"    {n_raw:,} raw -> {len(df):,} kept (excluded {n_raw - len(df):,})")
        qlog.append(dict(pool=name, query_used=used, used_fallback=fb,
                         n_raw=n_raw, n_after_exclusion=len(df), status="OK"))
        audit.append(dict(pool=name, n_raw=n_raw, n_kept=len(df)))

    pd.DataFrame(qlog).to_csv(f"{FETCHED}/uniprot_queries_used.csv", index=False)
    if audit:
        pd.DataFrame(audit).to_csv(f"{FETCHED}/negative_exclusion_audit.csv", index=False)



NONSTD_RE = "[BJOUXZ]"

CONTAMINANT_PATTERNS = {
    "ribosomal": r"\bribosomal\b",
    "casein": r"\bcasein\b",
    "lipoprotein": r"\blipoprotein\b|\bapolipoprotein\b",
    "immunoglobulin": r"\bimmunoglobulin\b|\big [a-z]\b",
    "rubisco": r"\brubisco\b|ribulose.bisphosphate carboxylase",
    "keratin": r"\bkeratin\b",
    "MFGM": r"milk fat globule",
    "histone": r"\bhistone\b",
    "haemoglobin": r"\bh(a)?emoglobin\b",
}


def flag_contaminant(protein_name):
    if not isinstance(protein_name, str):
        return 0, np.nan
    low = protein_name.lower()
    for cls, pat in CONTAMINANT_PATTERNS.items():
        if re.search(pat, low):
            return 1, cls
    return 0, np.nan


def loc_bucket(subcell, gocc):
    s = f"{subcell if isinstance(subcell, str) else ''} " \
        f"{gocc if isinstance(gocc, str) else ''}".lower().strip()
    if not s:
        return np.nan
    if re.search(r"secreted|extracellular", s):
        return "secreted_extracellular"
    if re.search(r"cell wall|apoplast", s):
        return "cell_wall_apoplast"
    if re.search(r"membrane", s):
        return "membrane"
    if re.search(r"nucleus|nuclear", s):
        return "nucleus"
    if re.search(r"mitochondri|chloroplast|plastid|peroxisom|vacuol|golgi|endoplasmic", s):
        return "organelle"
    if re.search(r"cytoplasm|cytosol", s):
        return "cytoplasm"
    return "other_annotated"


def parse_uniprot_block(df):
    out = pd.DataFrame(index=df.index)
    out["uniprot_id"] = df["Entry"]
    out["entry_name"] = df.get("Entry Name")
    out["protein_name"] = df.get("Protein names")
    out["gene_names"] = df.get("Gene Names")
    out["sequence"] = df.get("Sequence")
    out["reviewed"] = df.get("Reviewed")
    out["subcellular_location"] = df.get("Subcellular location [CC]")
    out["go_cc"] = df.get("Gene Ontology (cellular component)")
    sig = df.get("Signal peptide", pd.Series(index=df.index, dtype=object))
    tm = df.get("Transmembrane", pd.Series(index=df.index, dtype=object))
    out["signal_peptide"] = sig.notna().astype(int)
    out["n_transmembrane"] = tm.fillna("").astype(str).str.count("TRANSMEM")
    out["membrane_association"] = (out["n_transmembrane"] > 0).astype(int)
    out["secretion_status"] = out["signal_peptide"]
    pe = df.get("Protein existence", pd.Series(index=df.index, dtype=object))
    out["proteomic_detectability"] = pe.astype(str).str.contains(
        "protein level", case=False, na=False).astype(int)
    out["protein_existence"] = pe
    out["loc_bucket"] = [loc_bucket(a, b) for a, b in
                         zip(out["subcellular_location"], out["go_cc"])]
    return out


def coarse_stratum(df):
    loc = df["loc_bucket"].fillna("unannotated").astype(str)
    sec = pd.to_numeric(df["secretion_status"], errors="coerce").fillna(0).astype(int).astype(str)
    mem = pd.to_numeric(df["membrane_association"], errors="coerce").fillna(0).astype(int).astype(str)
    return "sec" + sec + "_mem" + mem + "_loc" + loc


def evidence_rank(df):
    not_reviewed = (df["reviewed"].astype(str).str.lower() != "reviewed").astype(int)
    not_protein_level = (pd.to_numeric(df["proteomic_detectability"], errors="coerce")
                          .fillna(0).astype(int) == 0).astype(int)
    return not_reviewed + not_protein_level


PAIRS = {"bovine_milk_EV": "bovine", "human_milk_EV": "human",
         "donkey_milk_EV": "donkey", "arabidopsis_apoplastic_EV": "arabidopsis",
         "brassica_apoplastic_EV": "brassica"}


def stage2_v3_build():
    pos = load_positives_raw()
    if not cached(ANN_OUT):
        sys.exit(f"[FATAL] {ANN_OUT} missing or empty -- re-run stage1_uniprot() "
                 f"until positives resolve successfully before running stage 2")
    ann_raw = pd.read_csv(ANN_OUT, low_memory=False)
    ann_cov = parse_uniprot_block(ann_raw)
    ann_cov["submitted_id"] = (ann_raw["From"].astype(str) if "From" in ann_raw.columns
                               else ann_cov["uniprot_id"].astype(str))
    ann_cov = ann_cov.drop_duplicates("submitted_id")

    KEEP_COV = ["subcellular_location", "go_cc", "signal_peptide", "n_transmembrane",
                "membrane_association", "secretion_status", "proteomic_detectability",
                "protein_existence", "loc_bucket", "reviewed"]
    pos["_join"] = pos["uniprot_id"].astype(str)
    pos = pos.merge(ann_cov[["submitted_id", "sequence", "protein_name", "gene_names"] + KEEP_COV]
                    .rename(columns={"submitted_id": "_join"}), on="_join", how="left")
    pos = pos.drop(columns=["_join"])

    # contaminant flagging -- rule-based, reported not deleted
    flags = pos["protein_name"].apply(flag_contaminant)
    pos["contaminant_flag"] = flags.apply(lambda t: t[0])
    pos["contaminant_class"] = flags.apply(lambda t: t[1])

    cascade = []
    for sc, d in pos.groupby("subclass"):
        d1 = d[d["sequence"].notna()]
        d2 = d1.drop_duplicates("sequence")
        d3 = d2[~d2["sequence"].astype(str).str.contains(NONSTD_RE, regex=True, na=False)]
        d4 = d3[d3["sequence"].astype(str).str.len().between(LEN_MIN, LEN_MAX)]
        for lbl, n in [("1_ids_from_source_study", len(d)), ("2_sequence_retrieved", len(d1)),
                       ("3_deduplicated_by_sequence", len(d2)), ("4_nonstandard_aa_removed", len(d3)),
                       (f"5_length_{LEN_MIN}_{LEN_MAX}", len(d4))]:
            cascade.append(dict(subclass=sc, step=lbl, n=n))

    pos_clean = (pos[pos["sequence"].notna()]
                 .drop_duplicates("sequence")
                 .loc[lambda d: ~d["sequence"].astype(str)
                      .str.contains(NONSTD_RE, regex=True, na=False)]
                 .loc[lambda d: d["sequence"].astype(str).str.len().between(LEN_MIN, LEN_MAX)]
                 .reset_index(drop=True))
    pos_clean["length"] = pos_clean["sequence"].astype(str).str.len()
    pos_clean["stratum_key"] = coarse_stratum(pos_clean)
    print(f"[stage2] positives after QC: {len(pos_clean):,}")
    print(pos_clean["subclass"].value_counts().to_string())

    neg_out, support_report, quota_report = [], [], []

    for subclass, gpos in pos_clean.groupby("subclass"):
        pool_name = PAIRS.get(subclass)
        path = f"{FETCHED}/negative_pool_{pool_name}.csv"
        if pool_name is None or not os.path.isfile(path):
            print(f"!! no pool for {subclass} -- SKIPPING")
            continue
        raw = pd.read_csv(path, low_memory=False)
        pool = parse_uniprot_block(raw)
        pool = (pool[pool["sequence"].notna()]
                .drop_duplicates("sequence")
                .loc[lambda d: ~d["sequence"].astype(str)
                     .str.contains(NONSTD_RE, regex=True, na=False)]
                .loc[lambda d: d["sequence"].astype(str).str.len().between(LEN_MIN, LEN_MAX)]
                .reset_index(drop=True))
        pool["length"] = pool["sequence"].astype(str).str.len()
        pool["stratum_key"] = coarse_stratum(pool)

        pos_strata = gpos["stratum_key"].value_counts()
        common = set(pos_strata.index)
        in_support = pool[pool["stratum_key"].isin(common)].copy()
        out_of_support = pool[~pool["stratum_key"].isin(common)]
        support_report.append(dict(
            subclass=subclass, pool=pool_name, pool_after_qc=len(pool),
            pool_in_common_support=len(in_support),
            pool_excluded_no_positive_analog=len(out_of_support),
            pct_excluded=round(100 * len(out_of_support) / max(1, len(pool)), 1),
            n_strata_occupied_by_positives=len(common)))

        picks = []
        for stratum, n_pos_s in pos_strata.items():
            quota = int(round(HEADROOM_MULTIPLE * n_pos_s))
            avail = in_support[in_support["stratum_key"] == stratum]
            n_avail = len(avail)
            if n_avail <= quota:
                chosen = avail.copy()
                note = "stratum_pool_exhausted_took_all"
            else:
                order = avail.assign(_rank=evidence_rank(avail)) \
                             .sample(frac=1.0, random_state=SEED) \
                             .sort_values("_rank", kind="stable")
                chosen = order.head(quota).drop(columns="_rank")
                note = "quota_met_evidence_preferred"
            chosen = chosen.copy()
            chosen["sampling_note"] = note
            picks.append(chosen)
            quota_report.append(dict(subclass=subclass, pool=pool_name, stratum=stratum,
                                     n_positive=int(n_pos_s), quota_at_headroom=quota,
                                     available_in_support=n_avail, drawn=len(chosen),
                                     shortfall=max(0, quota - n_avail)))

        neg = pd.concat(picks, ignore_index=True) if picks else in_support.iloc[0:0].copy()
        neg["polarity"] = "negative"
        neg["class_label"] = "Non_EV"
        neg["subclass"] = f"{subclass}__NEGATIVE"
        neg["matched_taxon_group"] = subclass
        neg["vesicle_category"] = "not_applicable"
        neg["evidence_tier"] = np.nan
        neg["evidence_tier_basis"] = (
            "negative class: taxonomy-matched UniProt protein with no EV evidence, "
            "drawn by frequency matching within the covariate stratum common to the "
            f"positive class (headroom x{HEADROOM_MULTIPLE} before joint CD-HIT)")
        neg["source_study_id"] = "N1_uniprot_negatives"
        neg["sample_type"] = "UniProt query"
        neg["organism"] = gpos["organism"].iloc[0]
        neg["taxon_id"] = gpos["taxon_id"].iloc[0]
        neg["kingdom"] = gpos["kingdom"].iloc[0]
        neg["species_group"] = gpos["species_group"].iloc[0]
        neg["negative_pool"] = pool_name
        if len(neg):
            nflags = neg["protein_name"].apply(flag_contaminant)
            neg["contaminant_flag"] = nflags.apply(lambda t: t[0])
            neg["contaminant_class"] = nflags.apply(lambda t: t[1])
        else:
            neg["contaminant_flag"], neg["contaminant_class"] = [], []
        neg["protein_uid"] = [f"NEG_{pool_name.upper()[:8]}_{i:05d}" for i in range(1, len(neg) + 1)]
        for c in pos_clean.columns:
            if c not in neg.columns:
                neg[c] = np.nan
        neg.to_csv(f"{OUT_BASE}/final/negatives/{subclass}_negative.csv", index=False)
        neg_out.append(neg)
        print(f"[NEG] {subclass:28s} pos={len(gpos):5d} neg_drawn={len(neg):5d} "
              f"pool_support={len(in_support):6d}/{len(pool):6d} strata={len(common)}")

    for sc, d in pos_clean.groupby("subclass"):
        d.to_csv(f"{OUT_BASE}/final/positives/{sc}_positive.csv", index=False)

    master = pd.concat([pos_clean] + neg_out, ignore_index=True) if neg_out else pos_clean

    DROP_COLS = ["abundance_ppm", "n_biological_replicates", "protein_id_threshold",
                 "has_nonstandard_aa", "length_pass", "orthogonal_evidence", "notes",
                 "purification_method", "len_bin", "matching_covariates_used",
                 "covariates_dropped_constant_in_pool", "matched_to_positive_subclass"]
    master = master.drop(columns=[c for c in DROP_COLS if c in master.columns])

    master.to_csv(f"{OUT_BASE}/final/FoodEVPred_v3_master.csv", index=False)
    pd.DataFrame(cascade).to_csv(f"{OUT_BASE}/final/reports/filtering_cascade.csv", index=False)
    pd.DataFrame(support_report).to_csv(f"{OUT_BASE}/final/reports/common_support_report.csv", index=False)
    pd.DataFrame(quota_report).to_csv(f"{OUT_BASE}/final/reports/stratum_sampling_report.csv", index=False)

    print(f"\n=== v3 MASTER (pre-clustering): {len(master):,} proteins ===")
    print(master.groupby(["polarity", "class_label"]).size().to_string())

    print("\n" + "=" * 78)
    print("NEXT STEPS")
    print("=" * 78)
    print("1. Joint 40%-identity CD-HIT (reviewer C6) across the whole master,")
    print("   positives and negatives together, one pass, cluster-head kept.")
    print("2. Whatever imbalance remains: class weighting at modelling time,")
    print("   report balanced accuracy / macro-F1 / PR-AUC alongside accuracy.")
    print("3. Re-run the 5 analyses reviewer C1 asked for on this rebuilt set.")



if __name__ == "__main__":
    stage0_load_positive_ids()
    stage1_uniprot()
    stage2_v3_build()


In [ ]:
# ── Install CD-HIT v4.8.1 ─────────────────────────────────────────────────
!wget -q https://github.com/weizhongli/cdhit/releases/download/V4.8.1/cd-hit-v4.8.1-2019-0228.tar.gz
!tar -xzf cd-hit-v4.8.1-2019-0228.tar.gz
!cd cd-hit-v4.8.1-2019-0228 && make -s

In [ ]:
#!/usr/bin/env python3
"""
FoodEVPred v3 -- joint 40%-identity CD-HIT

Runs CD-HIT once, jointly, across the entire v3 master (positives and
negatives together) at 40% sequence identity, and merges cluster
assignments back onto every row.

`uniprot_id` is used as the FASTA header and merge key throughout: it is
populated and globally unique across the entire master. The script exits
if that uniqueness assumption is ever violated, or if any row ends up
unmatched after clustering.

INPUT   /kaggle/working/foodevpred_v3/final/FoodEVPred_v3_master.csv
OUTPUT  /kaggle/working/foodevpred_v3/cdhit/
            foodev_all_classes.fasta
            clusters_foodev_0.4.fasta(.clstr)
        /kaggle/working/foodevpred_v3/final/
            FoodEVPred_v3_clustered.csv   <- master + cluster_id + is_representative, all rows
            FoodEVPred_v3_final.csv       <- representatives only; train/test split starts here
"""
import os, re, shutil, subprocess, sys
import pandas as pd

OUT_BASE = "/kaggle/working/foodevpred_v3"
MASTER = "/kaggle/input/datasets/harshiikkaa/foodevpred-v3/foodevpred_v3/final/FoodEVPred_v3_master.csv"
CDHIT_DIR = f"{OUT_BASE}/cdhit"
os.makedirs(CDHIT_DIR, exist_ok=True)

FASTA_IN = f"{CDHIT_DIR}/foodev_all_classes.fasta"
CLUST_OUT = f"{CDHIT_DIR}/clusters_foodev_0.4.fasta"
CLSTR_FILE = f"{CLUST_OUT}.clstr"

IDENTITY = 0.4
WORD_SIZE = 2   


def build_fasta():
    df = pd.read_csv(MASTER, low_memory=False)

    dup = df["uniprot_id"].duplicated().sum()
    if dup:
        sys.exit(f"[FATAL] {dup} duplicate uniprot_id in the master -- this MUST be "
                 f"unique, the merge-back after CD-HIT depends entirely on it. "
                 f"Fix the master before clustering.")

    n_missing_seq = df["sequence"].isna().sum()
    if n_missing_seq:
        print(f"[cdhit] !! {n_missing_seq} rows have no sequence -- excluded from "
              f"the FASTA (and therefore from clustering and the final dataset)")
    df = df[df["sequence"].notna()].copy()

    with open(FASTA_IN, "w") as f:
        for _, row in df.iterrows():
            f.write(f">{row['uniprot_id']}\n{row['sequence']}\n")
    print(f"[cdhit] wrote {len(df):,} sequences -> {FASTA_IN}")
    return df


def ensure_cdhit():
    """cd-hit is a compiled C++ tool, not a Python package; install it on
    demand if it isn't already on PATH."""
    if shutil.which("cd-hit"):
        print(f"[cdhit] found existing binary: {shutil.which('cd-hit')}")
        return

    print("[cdhit] 'cd-hit' not found on PATH -- installing via apt-get")
    for cmd in (["apt-get", "update", "-qq"],
                ["apt-get", "install", "-y", "-qq", "cd-hit"]):
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode != 0:
            print(f"  {' '.join(cmd)} -> exit {r.returncode}")
            print(r.stderr[-1000:])

    if shutil.which("cd-hit"):
        print(f"[cdhit] installed via apt-get: {shutil.which('cd-hit')}")
        return

    print("[cdhit] apt-get install failed -- trying conda")
    r = subprocess.run(["conda", "install", "-y", "-c", "bioconda", "cd-hit"],
                       capture_output=True, text=True)
    if shutil.which("cd-hit"):
        print(f"[cdhit] installed via conda: {shutil.which('cd-hit')}")
        return

    sys.exit(
        "[FATAL] could not install cd-hit automatically. Run ONE of these in a "
        "notebook cell yourself, then re-run this script (no need to redo build_fasta):\n"
        "  !apt-get update -qq && apt-get install -y cd-hit\n"
        "  !conda install -y -c bioconda cd-hit\n"
        "  or download a static binary from "
        "https://github.com/weizhongli/cdhit/releases and put it on PATH\n"
        "If apt-get itself fails with a network error, check that this notebook's "
        "internet access is turned on (Settings -> Internet -> On)."
    )


def run_cdhit():
    ensure_cdhit()
    cmd = ["cd-hit", "-i", FASTA_IN, "-o", CLUST_OUT,
           "-c", str(IDENTITY), "-n", str(WORD_SIZE),
           "-d", "0",     
           "-M", "0", "-T", "0",   # unlimited memory / use all available threads
           "-g", "1"]     # accurate mode: assign to most similar cluster, not first match
    print("[cdhit] running:", " ".join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout[-3000:])
    if result.returncode != 0:
        sys.exit(f"[FATAL] cd-hit failed (exit {result.returncode}):\n{result.stderr[-3000:]}")


def parse_clstr():
    """Parse the .clstr file into {uniprot_id: (cluster_id, is_representative)}."""
    assign = {}
    cluster_id = None
    with open(CLSTR_FILE) as f:
        for line in f:
            line = line.rstrip("\n")
            if line.startswith(">Cluster"):
                cluster_id = int(line.split()[-1])
                continue
            m = re.search(r">([^.]+)\.\.\.", line)
            if not m:
                continue
            uid = m.group(1)
            is_rep = line.rstrip().endswith("*")
            assign[uid] = (cluster_id, is_rep)
    return assign


def merge_and_report(df):
    assign = parse_clstr()
    df["cluster_id"] = df["uniprot_id"].map(lambda u: assign.get(u, (None, None))[0])
    df["is_representative"] = df["uniprot_id"].map(lambda u: assign.get(u, (None, None))[1])

    unmatched = df["cluster_id"].isna().sum()
    if unmatched:
        sys.exit(f"[FATAL] {unmatched} rows have no cluster assignment after the merge "
                 f"-- FASTA and master are out of sync, do not proceed with this output.")

    df.to_csv(f"{OUT_BASE}/final/FoodEVPred_v3_clustered.csv", index=False)


    mixed = (df.groupby("cluster_id")["class_label"].nunique() > 1).sum()
    print(f"\n[cdhit] clusters spanning >1 class_label: {mixed}  "
          f"(expected 0 -- any >0 means homologues split across classes)")

    rep = df[df["is_representative"] == True].copy()
    rep.to_csv(f"{OUT_BASE}/final/FoodEVPred_v3_final.csv", index=False)

    print(f"\n=== POST-CLUSTER (representatives only): {len(rep):,} sequences "
          f"(from {len(df):,} input, {len(df)-len(rep):,} collapsed as redundant) ===")
    print(rep.groupby(["polarity", "class_label", "subclass"]).size().to_string())
    print()
    print(rep.groupby("polarity").size().to_string())
    pos_n = int((rep.polarity == "positive").sum())
    neg_n = int((rep.polarity == "negative").sum())
    print(f"\nFinal negative:positive ratio = {neg_n / max(1, pos_n):.2f} : 1")


if __name__ == "__main__":
    df = build_fasta()
    run_cdhit()
    merge_and_report(df)

In [ ]:
import os

os.makedirs(f"{OUT_BASE}/final", exist_ok=True)
df.to_csv(f"{OUT_BASE}/final/FoodEVPred_v3_clustered.csv", index=False)

mixed = (df.groupby("cluster_id")["class_label"].nunique() > 1).sum()
print(f"[cdhit] clusters spanning >1 class_label: {mixed}")

rep = df[df["is_representative"] == True].copy()
rep.to_csv(f"{OUT_BASE}/final/FoodEVPred_v3_final.csv", index=False)

print(f"=== POST-CLUSTER (representatives only): {len(rep):,} sequences (from {len(df):,} input) ===")
print(rep.groupby(["polarity", "class_label", "subclass"]).size().to_string())
print()
print(rep.groupby("polarity").size().to_string())
pos_n = int((rep.polarity == "positive").sum())
neg_n = int((rep.polarity == "negative").sum())
print(f"Final negative:positive ratio = {neg_n / max(1, pos_n):.2f} : 1")

In [ ]:
#Install CD-HIT
!wget https://github.com/weizhongli/cdhit/releases/download/V4.8.1/cd-hit-v4.8.1-2019-0228.tar.gz
!tar -xzvf cd-hit-v4.8.1-2019-0228.tar.gz
%cd cd-hit-v4.8.1-2019-0228
!make
%cd ..

In [ ]:
import pandas as pd

MASTER = "/kaggle/input/datasets/harshiikkaa/foodevpred-v3/foodevpred_v3/final/FoodEVPred_v3_master.csv"
FASTA_IN = "/kaggle/working/foodev_all_classes.fasta"

df = pd.read_csv(MASTER, low_memory=False)
df = df[df["sequence"].notna()].copy()

with open(FASTA_IN, "w") as f:
    for _, row in df.iterrows():
        f.write(f">{row['uniprot_id']}\n{row['sequence']}\n")

print(f"wrote {len(df):,} sequences -> {FASTA_IN}")

In [ ]:
#Run CD-HIT
%cd cd-hit-v4.8.1-2019-0228
!./cd-hit -i /kaggle/working/foodev_all_classes.fasta -o /kaggle/working/clusters_foodev_0.4.fasta -c 0.4 -n 2
%cd ..

In [ ]:
import re


assign = {}
cluster_id = None
with open("/kaggle/working/clusters_foodev_0.4.fasta.clstr") as f:
    for line in f:
        line = line.rstrip("\n")
        if line.startswith(">Cluster"):
            cluster_id = int(line.split()[-1])
            continue
        m = re.search(r">([^.]+)\.\.\.", line)
        if not m:
            continue
        uid = m.group(1)
        assign[uid] = (cluster_id, line.rstrip().endswith("*"))

df["cluster_id"] = df["uniprot_id"].map(lambda u: assign.get(u, (None, None))[0])
df["is_representative"] = df["uniprot_id"].map(lambda u: assign.get(u, (None, None))[1])

unmatched = df["cluster_id"].isna().sum()
print(f"unmatched rows: {unmatched}")  